# PhotoPrism MLOps Infrastructure — Provisioning

This notebook provisions the infrastructure for the PhotoPrism MLOps project on Chameleon Cloud (KVM@TACC).

**What it creates:**
- 3 VM instances (node1, node2, node3) on KVM@TACC
- A private network (192.168.1.0/24) connecting all nodes
- Ports on sharednet1 with security groups (SSH, HTTP, service ports)
- A floating IP on node1 (the only public-facing node)

**Prerequisites:**
- Chameleon account with access to KVM@TACC
- `clouds.yaml` with valid application credentials uploaded to `/work/clouds.yaml`
- SSH key `id_rsa_chameleon` registered on Chameleon

**After provisioning, from your local terminal:**
1. Copy SSH key: `scp -i ~/.ssh/id_rsa_chameleon ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>:~/.ssh/id_rsa_chameleon`
2. SSH to node1: `ssh -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>`
3. Run setup: `git clone https://github.com/akashchauhanweb/photoprism-mlops-infra.git && bash photoprism-mlops-infra/scripts/setup_cluster.sh`

## Step 1: Configure Chameleon Context

In [ ]:
from chi import server, context, lease, network
import chi, os, time, datetime, subprocess, shutil, json

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

# ---- Project configuration ----
PROJECT_PREFIX = "proj24"
SSH_KEY_NAME   = "id_rsa_chameleon"
LEASE_HOURS    = 8
VM_FLAVOR      = "m1.medium"
VM_COUNT       = 3
VM_IMAGE       = "CC-Ubuntu24.04"

## Step 2: Install Terraform

In [ ]:
TF_VERSION = "1.14.4"

commands = [
    "mkdir -p /work/.local/bin",
    f"wget -q https://releases.hashicorp.com/terraform/{TF_VERSION}/terraform_{TF_VERSION}_linux_amd64.zip",
    f"unzip -o -q terraform_{TF_VERSION}_linux_amd64.zip",
    "mv terraform /work/.local/bin",
    f"rm terraform_{TF_VERSION}_linux_amd64.zip",
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {cmd}\n{result.stderr}")
    else:
        print(f"Done: {cmd}")

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]

result = subprocess.run("terraform --version", shell=True, capture_output=True, text=True)
print(result.stdout.split('\n')[0])

## Step 3: Create Lease

In [ ]:
LEASE_NAME = f"lease-infra-{PROJECT_PREFIX}"

l = lease.Lease(
    LEASE_NAME,
    duration=datetime.timedelta(hours=LEASE_HOURS),
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id(VM_FLAVOR),
    amount=VM_COUNT,
)
l.submit(idempotent=True)

reservation_id = l.get_reserved_flavors()[0].id
print(f"Lease: {LEASE_NAME} — Status: ACTIVE")
print(f"Reservation flavor ID: {reservation_id}")

## Step 4: Set Up Terraform Files

Creates the Terraform configuration adapted from the course lab IaC repo (`gourmetgram-iac`).

**Resources created by Terraform:**
- Private network + subnet (192.168.1.0/24, no gateway)
- 3 ports on private network (fixed IPs, no port security)
- 3 ports on sharednet1 (with security groups)
- 3 compute instances (CC-Ubuntu24.04, m1.medium)
- 1 floating IP assigned to node1

In [ ]:
tf_dir = "/work/photoprism-infra/tf/kvm"
os.makedirs(tf_dir, exist_ok=True)

# Copy clouds.yaml — Terraform reads credentials from here
shutil.copy("/work/clouds.yaml", f"{tf_dir}/clouds.yaml")

# Also place in default config location for openstack CLI
os.makedirs(os.path.expanduser("~/.config/openstack"), exist_ok=True)
shutil.copy("/work/clouds.yaml", os.path.expanduser("~/.config/openstack/clouds.yaml"))

# --- versions.tf ---
with open(f"{tf_dir}/versions.tf", 'w') as f:
    f.write("""terraform {
  required_version = \">= 0.14.0\"
  required_providers {
    openstack = {
      source  = \"terraform-provider-openstack/openstack\"
      version = \"~> 1.51.1\"
    }
  }
}
""")

# --- provider.tf ---
with open(f"{tf_dir}/provider.tf", 'w') as f:
    f.write("""provider \"openstack\" {
  cloud         = \"openstack\"
  endpoint_type = \"public\"
}
""")

# --- variables.tf ---
with open(f"{tf_dir}/variables.tf", 'w') as f:
    f.write("""variable \"suffix\" {
  description = \"Suffix for resource names (project ID)\"
  type        = string
  nullable    = false
}

variable \"key\" {
  description = \"Name of SSH key pair on Chameleon\"
  type        = string
  default     = \"id_rsa_chameleon\"
}

variable \"reservation\" {
  description = \"UUID of the reserved flavor\"
  type        = string
}

variable \"nodes\" {
  type = map(string)
  default = {
    \"node1\" = \"192.168.1.11\"
    \"node2\" = \"192.168.1.12\"
    \"node3\" = \"192.168.1.13\"
  }
}
""")

# --- data.tf ---
with open(f"{tf_dir}/data.tf", 'w') as f:
    f.write("""data \"openstack_networking_network_v2\" \"sharednet1\" {
  name = \"sharednet1\"
}

data \"openstack_networking_subnet_v2\" \"sharednet1_subnet\" {
  name = \"sharednet1-subnet\"
}

data \"openstack_networking_secgroup_v2\" \"allow_ssh\" {
  name = \"allow-ssh\"
}

data \"openstack_networking_secgroup_v2\" \"allow_http_80\" {
  name = \"allow-http-80\"
}

data \"openstack_networking_secgroup_v2\" \"allow_2342\" {
  name = \"allow-2342\"
}

data \"openstack_networking_secgroup_v2\" \"allow_8000\" {
  name = \"allow-8000\"
}

data \"openstack_networking_secgroup_v2\" \"allow_8080\" {
  name = \"allow-8080\"
}

data \"openstack_networking_secgroup_v2\" \"allow_9001\" {
  name = \"allow-9001\"
}

data \"openstack_networking_secgroup_v2\" \"allow_6333\" {
  name = \"allow-6333\"
}

data \"openstack_networking_secgroup_v2\" \"allow_30234\" {
  name = \"allow-30234\"
}

data \"openstack_networking_secgroup_v2\" \"allow_30500\" {
  name = \"allow-30500\"
}

data \"openstack_networking_secgroup_v2\" \"allow_30633\" {
  name = \"allow-30633-proj24\"
}
""")

# --- main.tf ---
with open(f"{tf_dir}/main.tf", 'w') as f:
    f.write("""# Private network for inter-node communication
resource \"openstack_networking_network_v2\" \"private_net\" {
  name                  = \"private-net-${var.suffix}\"
  port_security_enabled = false
}

resource \"openstack_networking_subnet_v2\" \"private_subnet\" {
  name       = \"private-subnet-${var.suffix}\"
  network_id = openstack_networking_network_v2.private_net.id
  cidr       = \"192.168.1.0/24\"
  no_gateway = true
}

# Ports on private network — fixed IPs, no port security
resource \"openstack_networking_port_v2\" \"private_net_ports\" {
  for_each              = var.nodes
  name                  = \"port-${each.key}-${var.suffix}\"
  network_id            = openstack_networking_network_v2.private_net.id
  port_security_enabled = false

  fixed_ip {
    subnet_id  = openstack_networking_subnet_v2.private_subnet.id
    ip_address = each.value
  }
}

# Ports on sharednet1 — with security groups for external access
resource \"openstack_networking_port_v2\" \"sharednet1_ports\" {
  for_each   = var.nodes
  name       = \"sharednet1-${each.key}-${var.suffix}\"
  network_id = data.openstack_networking_network_v2.sharednet1.id
  security_group_ids = [
    data.openstack_networking_secgroup_v2.allow_ssh.id,
    data.openstack_networking_secgroup_v2.allow_http_80.id,
    data.openstack_networking_secgroup_v2.allow_2342.id,
    data.openstack_networking_secgroup_v2.allow_8000.id,
    data.openstack_networking_secgroup_v2.allow_8080.id,
    data.openstack_networking_secgroup_v2.allow_9001.id,
    data.openstack_networking_secgroup_v2.allow_6333.id,
    data.openstack_networking_secgroup_v2.allow_30234.id,
    data.openstack_networking_secgroup_v2.allow_30500.id,
    data.openstack_networking_secgroup_v2.allow_30633.id,
  ]
}

# Compute instances
resource \"openstack_compute_instance_v2\" \"nodes\" {
  for_each = var.nodes

  name       = \"${each.key}-${var.suffix}\"
  image_name = \"CC-Ubuntu24.04\"
  flavor_id  = var.reservation
  key_pair   = var.key

  network {
    port = openstack_networking_port_v2.sharednet1_ports[each.key].id
  }

  network {
    port = openstack_networking_port_v2.private_net_ports[each.key].id
  }

  user_data = <<-EOF
    #! /bin/bash
    sudo echo \"127.0.1.1 ${each.key}-${var.suffix}\" >> /etc/hosts
    su cc -c /usr/local/bin/cc-load-public-keys
  EOF
}

# Floating IP — assigned to node1 only (jump host)
resource \"openstack_networking_floatingip_v2\" \"floating_ip\" {
  pool        = \"public\"
  description = \"PhotoPrism IP for ${var.suffix}\"
  port_id     = openstack_networking_port_v2.sharednet1_ports[\"node1\"].id
}
""")

# --- outputs.tf ---
with open(f"{tf_dir}/outputs.tf", 'w') as f:
    f.write("""output \"floating_ip\" {
  description = \"Floating IP assigned to node1\"
  value       = openstack_networking_floatingip_v2.floating_ip.address
}

output \"node_ips\" {
  description = \"Private network IPs for all nodes\"
  value       = { for k, v in var.nodes : k => v }
}
""")

print("Terraform files created:")
for f in sorted(os.listdir(tf_dir)):
    print(f"  {f}")

## Step 5: Terraform Init, Plan, and Apply

In [ ]:
# Build a clean environment — remove Chameleon Jupyter's OS_ vars
# which conflict with our clouds.yaml (they point to CHI@UC)
clean_env = {k: v for k, v in os.environ.items() if not k.startswith("OS_")}
clean_env["OS_CLOUD"]        = "openstack"
clean_env["PATH"]            = "/work/.local/bin:" + clean_env.get("PATH", "")
clean_env["HOME"]            = os.environ.get("HOME", "/home/jovyan")
clean_env["TF_VAR_suffix"]   = PROJECT_PREFIX
clean_env["TF_VAR_key"]      = SSH_KEY_NAME
clean_env["TF_VAR_reservation"] = reservation_id

def run_tf(command, description):
    print(f"\n{'='*60}")
    print(f"  {description}")
    print(f"{'='*60}")
    result = subprocess.run(
        command, shell=True, cwd=tf_dir, env=clean_env,
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise Exception(f"{description} failed with return code {result.returncode}")
    return result.returncode

run_tf("terraform init", "Terraform Init")
run_tf("terraform validate", "Terraform Validate")
run_tf("terraform plan", "Terraform Plan")

In [ ]:
# Apply — creates all resources
run_tf("terraform apply -auto-approve", "Terraform Apply")

# Extract outputs
result = subprocess.run(
    "terraform output -json",
    shell=True, cwd=tf_dir, env=clean_env,
    capture_output=True, text=True
)
outputs = json.loads(result.stdout)
floating_ip = outputs["floating_ip"]["value"]

print(f"\n{'='*60}")
print(f"  Infrastructure provisioned successfully!")
print(f"{'='*60}")
print(f"\n  Floating IP: {floating_ip}")
print(f"\n  Next steps from your local terminal:")
print(f"  1. scp -i ~/.ssh/id_rsa_chameleon ~/.ssh/id_rsa_chameleon cc@{floating_ip}:~/.ssh/id_rsa_chameleon")
print(f"  2. ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
print(f"  3. git clone https://github.com/akashchauhanweb/photoprism-mlops-infra.git && bash photoprism-mlops-infra/scripts/setup_cluster.sh")

---

## Teardown

Run these cells **only** when you want to destroy all infrastructure and free resources.

In [ ]:
# TEARDOWN: Destroy all Terraform-managed resources
# run_tf("terraform destroy -auto-approve", "Terraform Destroy")

In [ ]:
# TEARDOWN: Delete the lease
# l.delete()
# print("Lease deleted.")